# Train results — R0

**Strategy R0: whole-plan regeneration (baseline)**

This notebook reproduces the `R0` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R0` training run (`experiments/subset_train-r0`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R0-specific** metrics.
4. Derives final-outcome, repair-behaviour, phase and efficiency analyses from the run.
5. Writes the exact grouped table (Category | Metric | R0 TRAIN result) to `Train_results_R0.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Parameters — edit these if your run lives elsewhere

`STRATEGY` is this notebook's strategy. `RUN_DIR` is the **training run folder** to analyze. If a new/rerun `subset_train` experiment is saved under a different folder name, change `RUN_DIR` below and re-run the whole notebook — the two summary CSVs and the output `.xlsx` will regenerate from that run.

In [1]:
# === EDIT THESE if your training run is in a different folder ===
STRATEGY = 'R0'
RUN_DIR  = 'experiments/subset_train-r0'   # the subset_train run this notebook analyzes
# ================================================================

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [2]:
import subprocess, sys, os, csv
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)
assert 'STRATEGY' in globals() and 'RUN_DIR' in globals(), 'Run the Parameters cell first.'

# Analyzer output is written here, NEXT TO THIS NOTEBOOK (not a temp dir),
# so you can open overall_summary.csv / mechanism_summary.csv yourself.
OUT_DIR = str(NB_DIR / 'analysis_output')

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)
print('Analyzer output dir:', OUT_DIR)

Repo root : /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R0
Run folder: experiments/subset_train-r0
Analyzer output dir: /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output


In [3]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ /Users/enruifung/.pyenv/versions/3.12.8/bin/python3 scripts/analyze_repair_strategies.py --run R0=experiments/subset_train-r0 --output-dir /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output --no-figures


R0: source=experiments/subset_train-r0, declared/evidenced queries=60, attempt queries=59, proposals=331, matrix cells=360

Analyzer exit code: 0


### Step 0b — Keep only the two summary CSVs
The analyzer emits many files we don't use (and an empty `figures/` folder). This removes everything except `overall_summary.csv` and `mechanism_summary.csv`, so re-running never leaves clutter behind.

In [4]:
# Keep the analyzer output tidy: the analyzer writes ~18 files (+ an empty
# figures/ dir) per strategy, but this notebook only needs the two summary
# CSVs. Delete everything else so the folder holds just what's useful.
import shutil
strat_out = Path(OUT_DIR) / STRATEGY
KEEP = {'overall_summary.csv', 'mechanism_summary.csv'}
removed = 0
for item in strat_out.iterdir():
    if item.name in KEEP:
        continue
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()
    removed += 1
print(f'Cleaned {strat_out}: removed {removed} extra item(s); kept {sorted(KEEP)}')
print('Files now:', sorted(p.name for p in strat_out.iterdir()))

Cleaned /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output/R0: removed 16 extra item(s); kept ['mechanism_summary.csv', 'overall_summary.csv']
Files now: ['mechanism_summary.csv', 'overall_summary.csv']


## Step 1 — Load the analyzer's summary tables

> **Where do these CSVs come from?** They are **not** pre-existing files in the
> repo. **Step 0 above created them** by running the analyzer. They are written to
> `train_result_analysis/analysis_output/<STRATEGY>/` (printed by Step 0). If you
> look before running Step 0, the folder will not exist yet — that's expected.

`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [5]:
def load_summary(path):
    d = {}
    with open(path) as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 4

mechanism metrics available:
  plan_scope_proposals: count=331 valid_n=331 rate=1
  four_or_more_measured_changed_slots: count=53 valid_n=285 rate=0.185964912280702
  multiple_declared_personal_targets: count=36 valid_n=331 rate=0.108761329305136
  commonsense_proposals_regressing_personal: count=15 valid_n=144 rate=0.104166666666667


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [6]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

# (Category, Metric, Value). 'core' = metrics shared by every strategy.
core = [
    ('Dataset',          'Queries evaluated', queries),
    ('Repair behaviour', 'Total repair attempts', repair_attempts),
    ('Repair behaviour', 'Personal-constraint repair attempts', personal_attempts),
    ('Repair behaviour', 'Commonsense repair attempts', commonsense_attempts),
    ('Repair behaviour', 'Attempts that left the plan unchanged', f'{unchanged} ({unchanged_pct}%)'),
    ('Repair behaviour', 'Personal constraints fixed (across attempts)', personal_fixes),
    ('Repair behaviour', 'Personal constraints newly broken (across attempts)', personal_regressions),
]
for cat, k, v in core:
    print(f'  {cat:18} | {k:50} {v}')

  Dataset            | Queries evaluated                                  60
  Repair behaviour   | Total repair attempts                              331
  Repair behaviour   | Personal-constraint repair attempts                187
  Repair behaviour   | Commonsense repair attempts                        144
  Repair behaviour   | Attempts that left the plan unchanged              222 (67.07%)
  Repair behaviour   | Personal constraints fixed (across attempts)       14
  Repair behaviour   | Personal constraints newly broken (across attempts) 2


## Step 3 — R0-specific metrics
These describe R0's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`.

In [7]:
# R0-specific mechanism metrics (read from mechanism_summary.csv). These are
# shown for context/debugging but are NOT added to the final table, which is
# kept focused. 'specific' starts empty; the analysis cells below fill it.
def fmt_count_rate(row):
    c = int(float(row['count'])); n = int(float(row['valid_n'])); r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

MECH_KEYS = [
    ('Proposals that rewrote the whole plan', 'plan_scope_proposals'),
    ('Proposals that changed 4 or more slots', 'four_or_more_measured_changed_slots'),
    ('Attempts targeting multiple personal constraints at once', 'multiple_declared_personal_targets'),
    ('Commonsense repairs that broke a personal constraint', 'commonsense_proposals_regressing_personal'),
]
print('R0 mechanism metrics (context only, not in the table):')
for label, metric in MECH_KEYS:
    print(f'  {label:58} {fmt_count_rate(mech[metric]) if metric in mech else "n/a"}')

specific = []  # (category, metric, value) rows that WILL go below the spacer

R0 mechanism metrics (context only, not in the table):
  Proposals that rewrote the whole plan                      331 / 331 (100.0%)
  Proposals that changed 4 or more slots                     53 / 285 (18.6%)
  Attempts targeting multiple personal constraints at once   36 / 331 (10.9%)
  Commonsense repairs that broke a personal constraint       15 / 144 (10.4%)


In [8]:
# --- Load raw per-attempt and per-query data for analyses A-E ---
# (attempts.jsonl = one row per repair attempt; the per-query checkpoint JSON
#  carries initial_verification / final_verification and the repair_history.)
import json, glob
from collections import Counter

attempts = [json.loads(l) for l in open(Path(RUN_DIR, 'attempts.jsonl'), encoding='utf-8') if l.strip()]
ckpt_paths = sorted(glob.glob(str(Path(RUN_DIR, 'queries', '*', f'{STRATEGY}-*.json'))))
checkpoints = {}
for p in ckpt_paths:
    d = json.load(open(p, encoding='utf-8'))
    checkpoints[d['query_uid']] = d

def pc_pass_count(verif):
    # number of personal constraints that PASS (True) in a verification block
    pcs = (verif or {}).get('personal_constraints') or []
    return sum(1 for x in pcs if x is True)

print('attempts:', len(attempts), '| queries with checkpoints:', len(checkpoints))

attempts: 331 | queries with checkpoints: 60


### Final outcome — did the final plan beat the initial plan?
For each query, compare personal constraints passing in the **initial** vs **final** plan. Report how many queries improved, stayed the same, or worsened, and the net personal constraints gained (sum of per-query deltas).

In [9]:
improved = same = worsened = 0
delta_total = 0
for uid, d in checkpoints.items():
    pc_i = pc_pass_count(d.get('initial_verification'))
    pc_f = pc_pass_count(d.get('final_verification'))
    delta = pc_f - pc_i
    delta_total += delta
    if delta > 0: improved += 1
    elif delta < 0: worsened += 1
    else: same += 1
n_q = len(checkpoints)
def q(n): return f'{n}/{n_q} ({n / n_q * 100:.1f}%)'
outcome = [
    ('Final outcome', 'Queries improved', q(improved)),
    ('Final outcome', 'Queries unchanged', q(same)),
    ('Final outcome', 'Queries worsened', q(worsened)),
    ('Final outcome', 'Net personal constraints gained', delta_total),
]
specific += outcome
for cat, k, v in outcome: print(f'  {cat:14} | {k:40} {v}')

  Final outcome  | Queries improved                         5/60 (8.3%)
  Final outcome  | Queries unchanged                        47/60 (78.3%)
  Final outcome  | Queries worsened                         8/60 (13.3%)
  Final outcome  | Net personal constraints gained          -13


### Repair behaviour — what happens when R0 changes the plan?
Classify every attempt into mutually-exclusive outcomes using the measured slot change (`changed_slot_count`): confirmed no change (`0`), not measurable (`None`), or a confirmed change (`>0`) that either fixed ≥1 constraint without regression, introduced a regression (priority rule), or was neutral.

In [10]:
conf_unchanged = fixed_only = regressed = other = 0
for a in attempts:
    csc = a.get('changed_slot_count')
    if csc == 0:
        conf_unchanged += 1
    elif isinstance(csc, (int, float)) and csc > 0:
        if a.get('newly_violated'): regressed += 1
        elif a.get('fixed'): fixed_only += 1
        else: other += 1
    else:
        other += 1  # change not measurable
N = len(attempts)
def pr(n): return f'{n}/{N} ({n / N * 100:.1f}%)'
behaviour = [
    ('Repair behaviour', 'Attempts with confirmed no change', pr(conf_unchanged)),
    ('Repair behaviour', 'Changed and fixed ≥1 without regression', pr(fixed_only)),
    ('Repair behaviour', 'Changed and introduced a regression', pr(regressed)),
]
specific += behaviour
for cat, k, v in behaviour: print(f'  {cat:16} | {k:45} {v}')

  Repair behaviour | Attempts with confirmed no change             198/331 (59.8%)
  Repair behaviour | Changed and fixed ≥1 without regression       11/331 (3.3%)
  Repair behaviour | Changed and introduced a regression           16/331 (4.8%)


### Repair by phase (context only — not added to the focused table)
Shown for discussion; these rows are intentionally left out of the final table to keep it concise.

In [11]:
def phase_stats(phase):
    rows = [a for a in attempts if a.get('repair_phase') == phase]
    n = len(rows)
    if n == 0: return n, 'n/a', 'n/a'
    fix = sum(1 for a in rows if a.get('fixed'))
    viol = sum(1 for a in rows if a.get('newly_violated'))
    f = lambda x: f'{x}/{n} ({x / n * 100:.1f}%)'
    return n, f(fix), f(viol)
for phase in ('personal', 'commonsense'):
    n, fix, viol = phase_stats(phase)
    print(f'  {phase:11} attempts={n}  fixed>=1={fix}  new-violation={viol}')

  personal    attempts=187  fixed>=1=14/187 (7.5%)  new-violation=2/187 (1.1%)
  commonsense attempts=144  fixed>=1=0/144 (0.0%)  new-violation=15/144 (10.4%)


### Efficiency — how expensive is R0?
Average tokens, model calls, and repair time **per query** (tokens averaged only over queries with complete token data). R0 regenerates the whole itinerary, so this is the cost of that flexibility.

In [12]:
from collections import defaultdict
per_q_tokens = defaultdict(int); per_q_time = defaultdict(float)
per_q_complete = defaultdict(lambda: True)
for a in attempts:
    uid = a['query_uid']
    tok = a.get('total_tokens_delta')
    if tok is None: per_q_complete[uid] = False
    else: per_q_tokens[uid] += tok
    per_q_time[uid] += a.get('repair_time_seconds') or 0.0
all_q = list({a['query_uid'] for a in attempts})
complete_q = [u for u in all_q if per_q_complete[u]]
def avg(d, keys): return (sum(d[k] for k in keys) / len(keys)) if keys else 0
efficiency = [
    ('Efficiency', 'Average tokens per query', round(avg(per_q_tokens, complete_q))),
    ('Efficiency', 'Average repair time per query', f'{avg(per_q_time, all_q):.1f} s'),
]
specific += efficiency
for cat, k, v in efficiency: print(f'  {cat:12} | {k:35} {v}')

  Efficiency   | Average tokens per query            96849
  Efficiency   | Average repair time per query       96.0 s


### Example trajectory (context only — not added to the focused table)
A real attempt that fixed one personal constraint and broke another, printed for the write-up.

In [13]:
from chinatravel.evaluation.constraint_families import classify_constraint
def family_of(descriptors, idx):
    for d in descriptors or []:
        if d.get('constraint_index') == idx:
            try: fam = classify_constraint(d).get('family')
            except Exception: fam = None
            return fam or d.get('hard_logic_py', f'constraint {idx}')[:60]
    return f'constraint {idx}'
traj = next((a for a in attempts if a.get('fixed') and a.get('newly_violated')), None)
if traj is None:
    print('No attempt both fixed and newly-violated a constraint in this run.')
else:
    descs = traj.get('constraint_descriptors') or []
    uid = traj['query_uid']
    print(f'Query {uid}  (attempt step {traj["step"]}, phase {traj["repair_phase"]})')
    for i in traj['fixed']:
        print(f'    constraint {i} [{family_of(descs, i)}]: Fail -> Pass  (fixed)')
    for i in traj['newly_violated']:
        print(f'    constraint {i} [{family_of(descs, i)}]: Pass -> Fail  (new violation)')

Query e20241028160908981620  (attempt step 1, phase personal)
    constraint 3 [monetary_budget]: Fail -> Pass  (fixed)
    constraint 1 [trip_specification]: Pass -> Fail  (new violation)


## Step 4 — Assemble the full table (Category | Metric | R0 TRAIN result)

In [14]:
# Header + all core rows, a spacer, then all strategy-specific rows.
# Each data row is (Category, Metric, Value).
HEADER = ('Category', 'Metric', f'{STRATEGY} TRAIN result')
rows = [HEADER]
rows += core
rows += [('', '', '')]  # spacer: ABOVE = common to every strategy; BELOW = R0-specific
rows += specific

for cat, k, v in rows:
    print(f'{cat:16} | {k:60} | {v}')

Category         | Metric                                                       | R0 TRAIN result
Dataset          | Queries evaluated                                            | 60
Repair behaviour | Total repair attempts                                        | 331
Repair behaviour | Personal-constraint repair attempts                          | 187
Repair behaviour | Commonsense repair attempts                                  | 144
Repair behaviour | Attempts that left the plan unchanged                        | 222 (67.07%)
Repair behaviour | Personal constraints fixed (across attempts)                 | 14
Repair behaviour | Personal constraints newly broken (across attempts)          | 2
                 |                                                              | 
Final outcome    | Queries improved                                             | 5/60 (8.3%)
Final outcome    | Queries unchanged                                            | 47/60 (78.3%)
Final outcome    | Que

## Step 5 — Write the exact table to `Train_results_R0.xlsx`

In [15]:
import openpyxl
from openpyxl.styles import Font, Alignment

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (cat, k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, cat)
    ws.cell(r, 2, k)
    vcell = ws.cell(r, 3, v)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        vcell.number_format = '#,##0'  # show large numbers with thousands separators

# bold header row
for col in (1, 2, 3):
    ws.cell(1, col).font = Font(bold=True)
# wrap long metric text
for r in range(1, len(rows) + 1):
    ws.cell(r, 2).alignment = Alignment(wrap_text=True, vertical='top')
ws.column_dimensions['A'].width = 20
ws.column_dimensions['B'].width = 60
ws.column_dimensions['C'].width = 24

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/Train_results_R0.xlsx
Rows written: 18


## Step 6 — Read the file back to confirm it saved correctly

In [16]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        cat, k, v = (row + (None, None, None))[:3]
        print(f'{str(cat or ""):16} | {str(k or ""):58} | {v}')

Sheet: R0 train results | dims: A1:C18

Category         | Metric                                                     | R0 TRAIN result
Dataset          | Queries evaluated                                          | 60
Repair behaviour | Total repair attempts                                      | 331
Repair behaviour | Personal-constraint repair attempts                        | 187
Repair behaviour | Commonsense repair attempts                                | 144
Repair behaviour | Attempts that left the plan unchanged                      | 222 (67.07%)
Repair behaviour | Personal constraints fixed (across attempts)               | 14
Repair behaviour | Personal constraints newly broken (across attempts)        | 2
Final outcome    | Queries improved                                           | 5/60 (8.3%)
Final outcome    | Queries unchanged                                          | 47/60 (78.3%)
Final outcome    | Queries worsened                                           | 8/60 